# Week 5 · Demo — LLM-as-Judge ⚖️
### Score an answer the way a careful human would — with a rubric, at scale

Yesterday you built a **golden set** (questions + ideal answers). Today you build the thing that *uses* it: a **judge** that scores your app's answers against those ideals on a rubric — **accuracy, groundedness, format** — and explains its reasoning.

This is the generalization of the `is_accurate` function you hand-wrote on Day 1. Instead of coding a checker per question, you hand a **strong model** (`gpt-4o`) a rubric and let it judge *any* question.

**How to use this notebook:** run top to bottom. It runs **offline** with a deterministic stand-in judge (so the mechanics are free and reproducible), or **live** against `gpt-4o` by flipping one switch.

**Where this goes in your app:** the seed of `src/eval/judge.py`.

---
### Why a *strong* judge, and why `gpt-4o` (not `gpt-4o-mini`)
Judging is harder than answering — the judge has to tell a *subtly wrong* answer from a right one. A cheap judge can't, and a cheap judge that's confidently wrong is **worse than no judge**, because you'll trust its numbers. So: capstone answers use `gpt-4o-mini`; the **judge uses `gpt-4o`.**

## 0 · Setup — the golden entry, three candidate answers, and the judge switch

We reuse the remote-work entry from the golden-set demo. Then we define **three candidate answers** to judge — a good one, a weak one (missing the key detail), and a wrong one (contradicts the policy). A good judge should score them differently.

In [ ]:
import os, json

USE_FAKE = True   # <-- set False (and export OPENAI_API_KEY) to judge with the real gpt-4o

JUDGE_MODEL = "gpt-4o"     # the STRONG model — judging is where you don't cut corners

# one golden entry (from the golden-set demo)
golden = {
    "question": "How many days can I work remotely?",
    "ideal_answer": "Employees may work remotely up to 3 days per week, with manager approval.",
    "must_mention": ["3 days", "manager approval"],
}

# three candidate answers your /ask endpoint might have produced
candidates = {
    "good": "You may work remotely up to 3 days per week, with manager approval.",
    "weak": "Employees are allowed to work remotely with manager approval.",          # missing "3 days"
    "wrong": "There's no limit — you can work remotely whenever you like.",            # contradicts policy
}
print("judging", len(candidates), "candidates against 1 golden entry")

## 1 · The rubric prompt — telling the judge exactly what to look for

The judge is only as good as the rubric you give it. We spell out **three dimensions**, each on a **1–4 scale** (Poor / OK / Good / Excellent), and we give the judge the question, the *ideal* answer, and the candidate to score.

Coarse scale on purpose: a judge can reliably tell Poor from Good; it *cannot* reliably tell a 7 from an 8 on a 1–10 scale.

In [ ]:
RUBRIC = """You are a strict, fair evaluator of answers from a question-answering assistant.
Score the CANDIDATE answer against the IDEAL answer on three dimensions, each 1-4:

- accuracy    : are the facts correct and complete versus the ideal? (1 Poor .. 4 Excellent)
- groundedness: is it supported by the ideal/source, with nothing invented or contradictory?
- format      : is it clear, appropriately concise, and well-structured?

Scale: 1 = Poor, 2 = OK, 3 = Good, 4 = Excellent.
Be strict on accuracy: an answer that omits a key fact or contradicts the ideal cannot score above 2.
Return your scores and a one-paragraph reasoning that names specific facts."""

def build_messages(question, ideal, candidate):
    user = (f"QUESTION:\n{question}\n\n"
            f"IDEAL ANSWER:\n{ideal}\n\n"
            f"CANDIDATE ANSWER:\n{candidate}")
    return [{"role": "system", "content": RUBRIC},
            {"role": "user",   "content": user}]

print(build_messages(golden["question"], golden["ideal_answer"], candidates["good"])[1]["content"])

## 2 · Getting *structured* scores back — tool-calling (the W4 pattern, reused)

We don't want the judge to reply in free prose — we need machine-readable scores. So we reuse **tool-calling** from Week 4: declare a `JUDGE_TOOL` schema and force the judge to fill it. Same idea that gave your `/ask` its `confidence` field, now producing `{accuracy, groundedness, format, reasoning}`.

In [ ]:
JUDGE_TOOL = {
    "type": "function",
    "function": {
        "name": "submit_scores",
        "description": "Submit rubric scores and reasoning for the candidate answer.",
        "parameters": {
            "type": "object",
            "properties": {
                "accuracy":     {"type": "integer", "minimum": 1, "maximum": 4},
                "groundedness": {"type": "integer", "minimum": 1, "maximum": 4},
                "format":       {"type": "integer", "minimum": 1, "maximum": 4},
                "reasoning":    {"type": "string"},
            },
            "required": ["accuracy", "groundedness", "format", "reasoning"],
        },
    },
}

def _fake_judge(question, ideal, candidate, must_mention):
    """Deterministic stand-in so the notebook runs offline. A REAL judge reads for meaning."""
    c = candidate.lower()
    hits = [m for m in must_mention if m.lower() in c]
    contradicts = any(p in c for p in ["no limit", "unlimited", "whenever you like", "any day"])
    accuracy = 1 if contradicts else (4 if len(hits) == len(must_mention) else 2)
    grounded = 1 if contradicts else 4
    fmt = 4 if len(candidate.split()) <= 40 else 3
    reasoning = (f"Mentions {len(hits)}/{len(must_mention)} required facts {hits}. "
                 + ("Contradicts the policy (states there is no limit). " if contradicts else "")
                 + ("Missing a required fact. " if hits and len(hits) < len(must_mention) else ""))
    return {"accuracy": accuracy, "groundedness": grounded, "format": fmt, "reasoning": reasoning.strip()}

def judge(question, ideal, candidate, must_mention):
    if USE_FAKE:
        return _fake_judge(question, ideal, candidate, must_mention)
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=0,                       # judging should be as consistent as possible
        messages=build_messages(question, ideal, candidate),
        tools=[JUDGE_TOOL],
        tool_choice={"type": "function", "function": {"name": "submit_scores"}},
    )
    return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)

print("judge ready — mode:", "FAKE" if USE_FAKE else f"REAL ({JUDGE_MODEL})")

## 3 · Judge all three — and read the *reasoning*, not just the numbers

Run it. Watch the judge give the good answer high marks, dock the weak one on **accuracy** (missing "3 days"), and fail the wrong one on **accuracy and groundedness** (it contradicts the policy). The **reasoning** is where you check whether you agree with the judge — that's your spot-check.

In [ ]:
for label, candidate in candidates.items():
    s = judge(golden["question"], golden["ideal_answer"], candidate, golden["must_mention"])
    print(f"=== {label.upper()} ===")
    print(f"  candidate : {candidate}")
    print(f"  accuracy={s['accuracy']}  groundedness={s['groundedness']}  format={s['format']}")
    print(f"  reasoning : {s['reasoning']}\n")

Look at what the judge did that yesterday's hand-coded `is_accurate` needed custom code for — and it did it **from a rubric alone**, no per-question Python:
- **good** → high on all three.
- **weak** → dinged on accuracy for dropping the 3-day limit.
- **wrong** → failed accuracy *and* groundedness for contradicting the policy.

One judge, any question. That's the whole point — it *scales* where hand-written checkers don't.

## 4 · Why the 1–4 scale (and not 1–10)

Try changing the rubric to a 1–10 scale and re-judging (in real mode). You'll find the scores get **noisier and less repeatable** — the judge can't reliably defend a 7 vs an 8, so it wobbles. Four levels — Poor / OK / Good / Excellent — map to distinctions a judge can make *consistently*. Coarse scales trade false precision for reliability, and reliability is what you want from a measuring instrument.

(Notice the same four-level shape is used in your **Design Review rubric** — for the same reason.)

## 5 · The judge is not gospel — spot-check it

A judge is a strong instrument, not an oracle. Two disciplines keep it honest:
- **Use a strong judge** (`gpt-4o`). A `gpt-4o-mini` judge is confidently wrong and worse than none.
- **Spot-check ~10% by hand.** Read the reasoning; when you *disagree* with the judge, that's signal — either your rubric needs tightening, or the candidate was actually fine and you misjudged. Both are learning.

The reasoning field exists precisely so you *can* spot-check. A judge that only returns a number is a judge you can't audit.

## 6 · Your turn — experiment

1. **Go live.** `USE_FAKE = False`, export `OPENAI_API_KEY`, re-run. Compare the real `gpt-4o` reasoning to the canned one — the real judge reads for *meaning*, not substrings.
2. **Write a tricky candidate** that a substring check would pass but a real judge should fail (e.g. *"You may NOT work remotely 3 days a week."*). See whether the fake judge is fooled (it is — it's substring-based) and whether the real judge catches it (it should).
3. **Tighten the rubric.** Add a line like *"Deduct for any answer over 40 words."* Re-judge and watch the format scores move.
4. **Your domain.** Swap the golden entry and candidates for your capstone. The rubric barely changes — that's the power of judging from a rubric.

## Fit it into your app 🔧

This is the core of **`src/eval/judge.py`**:
- `JUDGE_TOOL` + `RUBRIC` + `build_messages` + `judge()` — lifted almost verbatim (drop the fake path or keep it behind `use_fake`).
- `JUDGE_MODEL = "gpt-4o"` — hard-code the strong judge; do **not** let it fall back to the capstone's `gpt-4o-mini`.

Then `scripts/run_eval.py` (Lab Step 3) will: load your 20 `golden_set.jsonl` entries → send each `question` to your **`/ask`** endpoint → `judge()` the candidate against the entry's `ideal_answer` → persist the scores to the `eval_runs` table → write aggregate stats to `docs/eval-run-001.md`. **That aggregate is your M1 eval baseline.**

**The one-line takeaway:** *a rubric + a strong judge turns "is this answer good?" from a per-question coding problem into a single, scalable evaluation you can run over your whole golden set.*